In [1]:
# libraries
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import math as m

from insolation import insolf # main package for terrain

import pygrib # NOT required
from netCDF4 import Dataset
from osgeo import gdal, osr

import random

In [5]:
UVU_OLSON_HOME = '/uufs/chpc.utah.edu/common/home/uvu-group1/olson/snow-data/'
HRRR_DATA = UVU_OLSON_HOME + 'HRRR_2022/'
ERW_TOPO_FILE = UVU_OLSON_HOME + 'isnobal-data/ERW_topo.nc'

In [41]:
## GDAL see attributes

hrrr_file = f"{HRRR_DATA}hrrr.20220301/hrrr.t16z.wrfsfcf06.grib2"
hh = gdal.Open(hrrr_file)
# Assume the first subDataSet holds the DEM info
# hh2 = gdal.Open(hh.GetSubDatasets()[0][0])
hh.GetRasterBand(10).GetMetadata().keys()

dict_keys(['GRIB_COMMENT', 'GRIB_DISCIPLINE', 'GRIB_ELEMENT', 'GRIB_FORECAST_SECONDS', 'GRIB_IDS', 'GRIB_PDS_PDTN', 'GRIB_PDS_TEMPLATE_ASSEMBLED_VALUES', 'GRIB_PDS_TEMPLATE_NUMBERS', 'GRIB_REF_TIME', 'GRIB_SHORT_NAME', 'GRIB_UNIT', 'GRIB_VALID_TIME'])

In [44]:
# CONVERT TO UTC TIME
# hh.GetRasterBand(10).GetMetadata()['GRIB_VALID_TIME']
from datetime import datetime

hrrr_file = f"{HRRR_DATA}hrrr.20220315/hrrr.t00z.wrfsfcf06.grib2"
hh = gdal.Open(hrrr_file)

# Extract the timestamp as an integer
unix_timestamp = int(hh.GetRasterBand(10).GetMetadata()['GRIB_VALID_TIME'])

# Convert it to a datetime object (in UTC)
readable_time = datetime.utcfromtimestamp(unix_timestamp)

print(readable_time)

2022-03-15 06:00:00


In [47]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo  # available in Python 3.9+

hrrr_file = f"{HRRR_DATA}hrrr.20220301/hrrr.t00z.wrfsfcf06.grib2"
hh = gdal.Open(hrrr_file)

# UTC timestamp from GRIB
unix_timestamp = int(hh.GetRasterBand(10).GetMetadata()['GRIB_VALID_TIME'])

# Create a UTC datetime
utc_dt = datetime.fromtimestamp(unix_timestamp, tz=timezone.utc)

# Convert to Colorado local time (with DST handling)
colorado_dt = utc_dt.astimezone(ZoneInfo("America/Denver"))

print("UTC time:", utc_dt)
print("Colorado time:", colorado_dt)


UTC time: 2022-03-01 06:00:00+00:00
Colorado time: 2022-02-28 23:00:00-07:00


In [45]:
# TEST topocalc function
#
from snobedo.shortwave import TopoShade
topo_file = ERW_TOPO_FILE
topo_shade = TopoShade(
        topo_file, TopoShade.SolarMethods.SKYFIELD
    )
topo_shade2 = TopoShade(
        topo_file, TopoShade.SolarMethods.SKYFIELD
    )
# JOES
time_range = [
        hrrr_dswrf.timestep_for_band(10) + timedelta(hours=hour)
        for hour in range(0, 24)
    ]
# OTHER
hour = hrrr_dswrf.timestep_for_band(10)
start_day = hour.strftime('%Y-%m-%d') 
end_day = (hour + timedelta(days=1)).strftime('%Y-%m-%d')
time_range2 = np.arange(str(start_day), str(end_day), np.timedelta64(1, 'h'), dtype='datetime64[s]')
time_range2 = [datetime.fromisoformat(str(r)) for r in time_range2]
print("Time Range: ")
print(time_range)
print("")
print("Time Range 2: ")
print(time_range2)
print("")
# 
# # # #
# calculate
topo_shade.calculate(time_range)
topo_shade2.calculate(time_range2)

topo_shade.zenith
topo_shade2.zenith

NameError: name 'TopoShade' is not defined

In [ ]:
import pygrib

# Open the GRIB file
hrrr_file = f"{HRRR_DATA}hrrr.20220301/hrrr.t16z.wrfsfcf06.grib2"
grbs = pygrib.open(hrrr_file)

grb = grbs.read(1)[0] # Reads the first message
print(grb) # Prints all metadata
print(grb.keys()) # Prints keys of metadata
print(grb['parameterName']) # Prints a specific metadata field

# for grb in grbs:
#     # Access metadata for the current message 'grb'
#     print(grb) # Prints all metadata
#     print(grb.keys()) # Prints keys of metadata
#     print(grb['parameterName']) # Prints a specific metadata field

# Print all available messages (metadata) in the GRIB file
# for grb in grbs:
    # print(grb)

# print(grb['dataType'])
# print(grb['level'])
# print(grb['forecastTime'])
# print(grb['validDate'])
# print(grb['Ni'], grb['Nj']) # grid dimensions


In [12]:
grb.keys()

['globalDomain',
 'GRIBEditionNumber',
 'tablesVersionLatestOfficial',
 'tablesVersionLatest',
 'grib2divider',
 'angleSubdivisions',
 'missingValue',
 'ieeeFloats',
 'isHindcast',
 'section0Length',
 'identifier',
 'discipline',
 'editionNumber',
 'totalLength',
 'sectionNumber',
 'section1Length',
 'numberOfSection',
 'centre',
 'centreDescription',
 'subCentre',
 'tablesVersion',
 'masterDir',
 'localTablesVersion',
 'significanceOfReferenceTime',
 'year',
 'month',
 'day',
 'hour',
 'minute',
 'second',
 'dataDate',
 'julianDay',
 'dataTime',
 'productionStatusOfProcessedData',
 'typeOfProcessedData',
 'md5Section1',
 'selectStepTemplateInterval',
 'selectStepTemplateInstant',
 'stepType',
 'is_chemical',
 'is_chemical_distfn',
 'is_chemical_srcsink',
 'is_aerosol',
 'is_aerosol_optical',
 'setCalendarId',
 'deleteCalendarId',
 'sectionNumber',
 'grib2LocalSectionPresent',
 'deleteLocalDefinition',
 'sectionNumber',
 'gridDescriptionSectionPresent',
 'section3Length',
 'numberOfSec